In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [9]:
datasets = {
    "Bursty": [
        {"id": "exp_20250625_014444", "name": "OpenWhisk"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
        {"id": "exp_20250624_210136", "name": "Histogram"},
        {"id": "exp_20250624_165438", "name": "Pagurus"},
    ],
    "Normal": [
        {"id": "exp_20250626_205804", "name": "OpenWhisk"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
        {"id": "exp_20250626_155721", "name": "Histogram"},
        {"id": "exp_20250626_113035", "name": "Pagurus"},
    ],
    "Similar": [
        {"id": "exp_20250625_162425", "name": "OpenWhisk"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
        {"id": "exp_20250625_211541", "name": "Histogram"},
        {"id": "exp_20250626_050332", "name": "Pagurus"},
    ],
}

In [10]:
experiments = [
    {"id": "exp_20250627_145610", "name": "Openwhisk"},
    {"id": "exp_20250627_103341", "name": "NMIG"},
    {"id": "exp_20250627_213220", "name": "Histogram"},
    {"id": "exp_20250628_065241", "name": "Pagurus"},
]

In [11]:
# import pandas as pd
# import json

def parse_json_safe(val):
    if pd.isna(val) or val == "None":
        return None
    if isinstance(val, dict):  # Already a dict
        return val
    try:
        return json.loads(val)
    except Exception:
        return None

def extract_name(res,name):
    if isinstance(res, dict):
        return res.get(name)
    return None




def extract_error_message(res):
    if isinstance(res, dict):
        resp = res.get("response", {})
        if not resp.get("success", True):
            return resp.get("result", {}).get("error")
    return None

def has_cuda_error(msg):
    if isinstance(msg, str):
        return "cuda" in msg.lower() or "cudnn" in msg.lower()
    return False

def extract_success(res):
    if isinstance(res, dict):
        return res.get("response", {}).get("success", True)
    return False



In [12]:
# plt.figure(figsize=(4, 3.5))
results = {}
df_res = {}

def pick_particular_column(rp,col="initTime"):
    # rp is expected to be a dict with key 'annotations' that is a list of dicts
    if not isinstance(rp, dict):
        return 0
    items = rp.get("annotations", []) or []
    for it in items:
        if isinstance(it, dict) and it.get("key") == col:
            return (it.get("value", 0))/1000
    return 0

    
for exp in experiments:
    # load the CSV for this experiment
    path = f"../results/{exp['id']}/results_updated.csv"
    df = pd.read_csv(path)

    # parse JSON and compute latency
    df["result_parsed"] = df["result"].apply(parse_json_safe)
    df["name"] = df["result_parsed"].apply(lambda res: extract_name(res, "name"))
    df['name'] = df['name'].str.replace(r'_p$', '', regex=True)
    df["start"] = df["result_parsed"].apply(lambda res: extract_name(res, "start"))
    df["end"] = df["result_parsed"].apply(lambda res: extract_name(res, "end"))
    df['latency'] = (df['end'] - df['start'])/1000
    df["error_message"] = df["result_parsed"].apply(extract_error_message)
    df["has_cuda_or_cudnn_error"] = df["error_message"].apply(has_cuda_error)
    df["success"] = df["result_parsed"].apply(extract_success)
    df["error"] = df["result_parsed"].isna() | (~df["success"])
    df["initTime"] = df["result_parsed"].apply(lambda x: pick_particular_column(x, col="initTime"))
    df["waitTime"] = df["result_parsed"].apply(lambda x: pick_particular_column(x, col="waitTime"))
    

    df_res[exp['name']] = df
    # error_counts = df['error'].value_counts()
    # results[exp['name']] = {}
    # results[exp['name']]['true_val'] = error_counts.get(True, 0)
    # results[exp['name']]['false_val'] = error_counts.get(False, 0)
    # results[exp['name']]['false_val'] = df['latency'].mean()
    # df_valid = df[df["error"] != True]
    # avg_latency_per_name_dict = df_valid.groupby("name")["latency"].mean().to_dict()
    # print(avg_latency_per_name_dict)
    # df_res[exp['
    print(exp)
    
   
    # df = df[df['latency'] < 300]

print(results)

{'id': 'exp_20250627_145610', 'name': 'Openwhisk'}
{'id': 'exp_20250627_103341', 'name': 'NMIG'}
{'id': 'exp_20250627_213220', 'name': 'Histogram'}
{'id': 'exp_20250628_065241', 'name': 'Pagurus'}
{}


In [13]:
# for l in experiments:
# count_zero = (df_res['Openwhisk']["initTime"] == 0).sum()

for l in df_res:
    count_zero_initTime = (df_res[l]["initTime"] == 0).sum()
    print(f"{l} - {count_zero_initTime}")
print(20*"*")
for l in df_res:
    count_zero_initTime = (df_res[l]["waitTime"] == 0).sum()
    print(f"{l} - {count_zero_initTime}")

Openwhisk - 734
NMIG - 660
Histogram - 650
Pagurus - 733
********************
Openwhisk - 0
NMIG - 0
Histogram - 0
Pagurus - 0


In [14]:
import pandas as pd
import json
from pathlib import Path

datasets = {
    "Bursty": [
        {"id": "exp_20250625_014444", "name": "OpenWhisk"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
        {"id": "exp_20250624_210136", "name": "Histogram"},
        {"id": "exp_20250624_165438", "name": "Pagurus"},
    ],
    "Normal": [
        {"id": "exp_20250626_205804", "name": "OpenWhisk"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
        {"id": "exp_20250626_155721", "name": "Histogram"},
        {"id": "exp_20250626_113035", "name": "Pagurus"},
    ],
    "Similar": [
        {"id": "exp_20250625_162425", "name": "OpenWhisk"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
        {"id": "exp_20250625_211541", "name": "Histogram"},
        {"id": "exp_20250626_050332", "name": "Pagurus"},
    ],
}


def parse_json_safe(val):
    if pd.isna(val) or val == "None":
        return None
    if isinstance(val, dict):
        return val
    try:
        return json.loads(val)
    except Exception:
        return None


def extract_name(res, name):
    if isinstance(res, dict):
        return res.get(name)
    return None


def extract_error_message(res):
    if isinstance(res, dict):
        resp = res.get("response", {})
        if not resp.get("success", True):
            return resp.get("result", {}).get("error")
    return None


def has_cuda_error(msg):
    if isinstance(msg, str):
        return "cuda" in msg.lower() or "cudnn" in msg.lower()
    return False


def extract_success(res):
    if isinstance(res, dict):
        return res.get("response", {}).get("success", True)
    return False


def pick_particular_column(rp, col="initTime"):
    if not isinstance(rp, dict):
        return 0

    items = rp.get("annotations", []) or []

    for it in items:
        if isinstance(it, dict) and it.get("key") == col:
            return it.get("value", 0) / 1000

    return 0


def process_experiment(exp, dataset_name):
    path = Path(f"../results/{exp['id']}/results_updated.csv")

    if not path.exists():
        print(f"File not found: {path}")
        return None

    df = pd.read_csv(path)

    df["dataset"] = dataset_name
    df["method"] = exp["name"]
    df["experiment_id"] = exp["id"]

    df["result_parsed"] = df["result"].apply(parse_json_safe)

    df["name"] = df["result_parsed"].apply(lambda res: extract_name(res, "name"))
    df["name"] = df["name"].astype(str).str.replace(r"_p$", "", regex=True)

    df["start"] = df["result_parsed"].apply(lambda res: extract_name(res, "start"))
    df["end"] = df["result_parsed"].apply(lambda res: extract_name(res, "end"))

    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"] = pd.to_numeric(df["end"], errors="coerce")

    df["latency"] = (df["end"] - df["start"]) / 1000

    df["error_message"] = df["result_parsed"].apply(extract_error_message)
    df["has_cuda_or_cudnn_error"] = df["error_message"].apply(has_cuda_error)

    df["success"] = df["result_parsed"].apply(extract_success)
    df["error"] = df["result_parsed"].isna() | (~df["success"])

    df["initTime"] = df["result_parsed"].apply(
        lambda x: pick_particular_column(x, col="initTime")
    )

    df["waitTime"] = df["result_parsed"].apply(
        lambda x: pick_particular_column(x, col="waitTime")
    )

    return df


df_res = {}
all_dfs = []

for dataset_name, experiments in datasets.items():
    df_res[dataset_name] = {}

    for exp in experiments:
        df = process_experiment(exp, dataset_name)

        if df is not None:
            df_res[dataset_name][exp["name"]] = df
            all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

print(combined_df.head())

       timestamp                                               func  \
0  1750783517000  556ccf8758c8c2a20082c161e955405e950439f0503522...   
1  1750783517000  556ccf8758c8c2a20082c161e955405e950439f0503522...   
2  1750783517000  556ccf8758c8c2a20082c161e955405e950439f0503522...   
3  1750783517000  556ccf8758c8c2a20082c161e955405e950439f0503522...   
4  1750783517000  556ccf8758c8c2a20082c161e955405e950439f0503522...   

                      activation_id  \
0  0084357ef63a4f6784357ef63a6f6734   
1  a0b7754e7eff4cdcb7754e7effacdcfe   
2  0248507609cc4c8e88507609cc5c8e37   
3  d7677c88e0bf4997a77c88e0bf999744   
4  79850cc3922a492c850cc3922ad92c52   

                                              result dataset     method  \
0  {\n    "namespace": "guest",\n    "name": "eff...  Bursty  OpenWhisk   
1  {\n    "namespace": "guest",\n    "name": "eff...  Bursty  OpenWhisk   
2  {\n    "namespace": "guest",\n    "name": "eff...  Bursty  OpenWhisk   
3  {\n    "namespace": "guest",\n    "

In [15]:
summary = (
    combined_df
    .groupby(["dataset", "method", "experiment_id"])
    .agg(
        total_requests=("latency", "count"),
        successful_requests=("success", "sum"),
        failed_requests=("error", "sum"),
        cuda_errors=("has_cuda_or_cudnn_error", "sum"),
        avg_latency_s=("latency", "mean"),
        median_latency_s=("latency", "median"),
        p95_latency_s=("latency", lambda x: x.quantile(0.95)),
        avg_init_time_s=("initTime", "mean"),
        avg_wait_time_s=("waitTime", "mean"),
        total_latency_s=("latency", "sum"),
    )
    .reset_index()
)

summary

,dataset,method,experiment_id,total_requests,successful_requests,failed_requests,cuda_errors,avg_latency_s,median_latency_s,p95_latency_s,avg_init_time_s,avg_wait_time_s,total_latency_s
0,Bursty,Histogram,exp_20250624_210136,996,957,39,0,2.494671,0.9365,8.31275,0.367547,0.703628,2484.692
1,Bursty,NMIG,exp_20250625_071954,996,996,0,0,4.916197,3.1265,12.64075,0.053647,0.761553,4896.532
2,Bursty,OpenWhisk,exp_20250625_014444,996,977,19,0,2.707020,0.9400,9.77350,0.318598,0.643913,2696.192
3,Bursty,Pagurus,exp_20250624_165438,996,975,21,0,2.865359,0.9410,14.29925,0.327155,0.658474,2853.898
4,Normal,Histogram,exp_20250626_155721,903,864,39,0,2.588971,1.0770,13.61840,0.332173,0.516553,2337.841
5,Normal,NMIG,exp_20250627_042542,922,922,0,0,4.943939,3.2180,15.14870,0.042613,0.642944,4558.312
6,Normal,OpenWhisk,exp_20250626_205804,922,900,22,0,3.414350,1.0995,15.67070,0.290062,0.627529,3148.031
7,Normal,Pagurus,exp_20250626_113035,921,913,8,0,3.304515,1.1180,15.55600,0.265714,0.594336,3043.458
8,Similar,Histogram,exp_20250625_211541,1079,961,118,0,0.944425,0.6390,2.35830,0.073253,0.154465,1019.035
9,Similar,NMIG,exp_20250625_113106,1286,1286,0,0,3.102970,2.9755,5.03500,0.013370,0.136061,3990.420


In [17]:
summary.to_csv("three_dataset_summary_table.csv", index=False)